# NeuroVerse — Colab Demo Backend

Google Colab cannot properly host a full Next.js app with a browser UI, sessions, and file uploads — that part of NeuroVerse should run locally in VSCode (see the project's `README.md`).

What this notebook **does** give you: a standalone FastAPI backend, exposed publicly through `ngrok`, that implements the same core NeuroVerse logic as the main app and talks to the **same NeonDB database**:
- skill-level classification from assessment scores (with thresholds)
- session pricing (base rate × skill-level factor × difficulty factor × mentor-expertise factor, +10% platform fee)
- mentor suitability matching %

Use this to test/demo the algorithms, call the API from a phone/browser, or plug into a Colab-based data-science workflow (e.g. analyzing mentor ratings). For the real product UI, run the VSCode project.

In [ ]:
!pip install fastapi uvicorn pyngrok psycopg2-binary python-multipart nest_asyncio -q

## 1. Configure your NeonDB connection
Paste the same `DATABASE_URL` you used in the VSCode project's `.env.local`. Get it from https://console.neon.tech

In [ ]:
import os
from getpass import getpass

DATABASE_URL = getpass('Paste your NeonDB DATABASE_URL: ')
NGROK_AUTHTOKEN = getpass('Paste your ngrok authtoken (from https://dashboard.ngrok.com/get-started/your-authtoken), or leave blank: ')
os.environ['DATABASE_URL'] = DATABASE_URL

## 2. Core logic (mirrors lib/classification.js, lib/pricing.js, lib/matching.js)

In [ ]:
# --- Classification thresholds ---
ADVANCED_MIN = 80
INTERMEDIATE_MIN = 50

def classify_level(score_percent: float) -> str:
    if score_percent >= ADVANCED_MIN:
        return 'advanced'
    if score_percent >= INTERMEDIATE_MIN:
        return 'intermediate'
    return 'beginner'

# --- Pricing engine ---
DEFAULT_BASE_RATE = 20  # credits/hour
PLATFORM_FEE_RATE = 0.10
SKILL_LEVEL_FACTORS = {'beginner': 0.5, 'intermediate': 1.0, 'advanced': 1.5}

def mentor_expertise_multiplier(avg_rating: float, total_reviews: int) -> float:
    if not total_reviews:
        return 1.0  # neutral for brand new mentors
    if avg_rating >= 4.5:
        return 1.5
    if avg_rating >= 3.5:
        return 1.2
    if avg_rating >= 2.0:
        return 1.0
    return 0.5  # < 2 rating

def compute_session_price(mentor_level, skill_difficulty_factor, mentor_avg_rating, mentor_total_reviews, hours, base_rate=DEFAULT_BASE_RATE):
    level_factor = SKILL_LEVEL_FACTORS.get(mentor_level, 1.0)
    expertise_factor = mentor_expertise_multiplier(mentor_avg_rating, mentor_total_reviews)
    price_per_hour = round(base_rate * level_factor * skill_difficulty_factor * expertise_factor, 2)
    base_credits = round(price_per_hour * hours, 2)
    platform_fee = round(base_credits * PLATFORM_FEE_RATE, 2)
    total_to_learner = round(base_credits + platform_fee, 2)
    credited_to_mentor = round(base_credits - platform_fee, 2)
    return {
        'price_per_hour': price_per_hour,
        'base_credits': base_credits,
        'platform_fee': platform_fee,
        'total_charged_to_learner': total_to_learner,
        'credited_to_mentor': credited_to_mentor,
    }

# --- Mentor suitability matching ---
LEVEL_ORDER = {'beginner': 1, 'intermediate': 2, 'advanced': 3}

def compute_suitability(mentor_level, target_level, mentor_languages, learner_languages, mentor_rating):
    score = 0
    mentor_rank = LEVEL_ORDER.get(mentor_level, 1)
    target_rank = LEVEL_ORDER.get(target_level, 1)
    if mentor_rank >= target_rank:
        score += 20 if mentor_rank == target_rank else 17
    overlap = set(mentor_languages or []) & set(learner_languages or [])
    score += 15 if overlap else (7.5 if not learner_languages else 0)
    score += min(mentor_rating / 5, 1) * 15
    score += 40  # skill match assumed true when filtering candidates by skill_id upstream
    score += 5   # neutral availability credit
    return round(min(score, 100))

print('Core logic loaded.')

## 3. FastAPI app exposing the logic + NeonDB queries

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import psycopg2
import psycopg2.extras

app = FastAPI(title='NeuroVerse Colab Demo API')

def get_conn():
    return psycopg2.connect(DATABASE_URL, sslmode='require')

@app.get('/health')
def health():
    return {'status': 'ok'}

class ScoreRequest(BaseModel):
    score_percent: float

@app.post('/classify')
def classify(req: ScoreRequest):
    return {'score_percent': req.score_percent, 'level': classify_level(req.score_percent)}

class PriceRequest(BaseModel):
    mentor_level: str
    skill_difficulty_factor: float = 1.0
    mentor_avg_rating: float = 0
    mentor_total_reviews: int = 0
    hours: float = 1
    base_rate: float = DEFAULT_BASE_RATE

@app.post('/price')
def price(req: PriceRequest):
    return compute_session_price(req.mentor_level, req.skill_difficulty_factor, req.mentor_avg_rating, req.mentor_total_reviews, req.hours, req.base_rate)

@app.get('/skills')
def list_skills():
    conn = get_conn()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute('SELECT id, name, category, test_type, difficulty_factor FROM skills ORDER BY name')
    rows = cur.fetchall()
    conn.close()
    return {'skills': rows}

@app.get('/mentors/{skill_id}')
def mentors_for_skill(skill_id: str, target_level: str = 'intermediate'):
    conn = get_conn()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute('''
        SELECT c.id as course_id, c.title, c.mentor_level, u.name as mentor_name, u.languages,
               mr.avg_rating, mr.total_reviews
        FROM courses c
        JOIN users u ON c.mentor_id = u.id
        LEFT JOIN mentor_reputation mr ON mr.mentor_id = c.mentor_id
        WHERE c.skill_id = %s
    ''', (skill_id,))
    rows = cur.fetchall()
    conn.close()
    for r in rows:
        r['suitability_percent'] = compute_suitability(
            r['mentor_level'], target_level, r['languages'], [], float(r['avg_rating'] or 0)
        )
    rows.sort(key=lambda r: r['suitability_percent'], reverse=True)
    return {'mentors': rows}

print('FastAPI app ready.')

## 4. Launch the API publicly with ngrok

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok, conf

nest_asyncio.apply()

if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN

public_url = ngrok.connect(8000)
print('Public API URL:', public_url)
print('Try it:', str(public_url) + '/skills')

uvicorn.run(app, host='0.0.0.0', port=8000)

## Notes
- This notebook is a **testing/demo companion**, not a replacement for the full app.
- Run the real NeuroVerse product (signup/login, onboarding, assessments, booking, live session room with chat/files/Gemini assistant, wallet, dashboard) from the VSCode project using the steps in `README.md`.
- Both this notebook and the VSCode app can point at the **same NeonDB** database, so data (skills, mentors, ratings) stays in sync between them.
- Stop the `uvicorn.run(...)` cell (Runtime → Interrupt execution) when you're done, or the ngrok tunnel will keep running until the Colab session ends.